# LAB-HW-03 — 第一次生成并配置 bitstream

**今天只解决一个问题：把一段极小 RTL 真正变成 KV260 PL 中运行的配置。**

前置：LAB-HW-00~02 已通过。今天不学习 Linux、AXI，也不重新学习 neuron 算法。

**Project Trace:** RMD-012A · T-HW-003/T-HW-011

## 1. 这次到底要让 FPGA 做什么

课程提供的 `kv260_marker_top` 只有一个行为：

```systemverilog
bank45_gpio = 5'b10101;
```

没有 clock，没有 reset，没有 host communication。

这样，今天如果失败，问题只可能落在 **build / implementation / bitstream / JTAG programming / physical mapping** 这一小段，而不是神经网络逻辑。

<svg xmlns="http://www.w3.org/2000/svg" width="820" height="250" viewBox="0 0 820 250" role="img" aria-label="LAB-HW-03 first bitstream flow">
  <rect x="20" y="80" width="130" height="70" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="85" y="110" text-anchor="middle" font-size="15">marker RTL</text>
  <text x="85" y="133" text-anchor="middle" font-size="12">5'b10101</text>
  <rect x="190" y="80" width="130" height="70" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="255" y="110" text-anchor="middle" font-size="15">Vivado</text>
  <text x="255" y="133" text-anchor="middle" font-size="12">synth + impl</text>
  <rect x="360" y="80" width="130" height="70" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="425" y="110" text-anchor="middle" font-size="15">bitstream</text>
  <text x="425" y="133" text-anchor="middle" font-size="12">.bit + reports</text>
  <rect x="530" y="80" width="130" height="70" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="595" y="110" text-anchor="middle" font-size="15">JTAG program</text>
  <text x="595" y="133" text-anchor="middle" font-size="12">xck26*</text>
  <rect x="700" y="80" width="100" height="70" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="750" y="110" text-anchor="middle" font-size="14">Bank45</text>
  <text x="750" y="133" text-anchor="middle" font-size="12">visible marker</text>
  <path d="M150 115 L190 115 M320 115 L360 115 M490 115 L530 115 M660 115 L700 115" stroke="#333" stroke-width="2"/>
  <polygon points="190,115 180,110 180,120" fill="#333"/><polygon points="360,115 350,110 350,120" fill="#333"/>
  <polygon points="530,115 520,110 520,120" fill="#333"/><polygon points="700,115 690,110 690,120" fill="#333"/>
</svg>

## 2. 为什么可以用这 5 个 output

AMD/Xilinx Board Store 的 KV260 carrier board data 把 `bank45_gpio` 描述成 5-bit、LED-class GPIO output。K26 SOM pin map 给出了对应 package pins：

| bit | SOM240 | package pin |
|---|---|---|
| 0 | D18 | J11 |
| 1 | B17 | J10 |
| 2 | B18 | K13 |
| 3 | A15 | F11 |
| 4 | C24 | A12 |

课程把这些 mapping 放进 `boards/kv260/constraints/bank45_gpio.xdc`。

**今天不要改 XDC。** 今天先把它当成已经审查过的 board adapter；LAB-HW-04 再解释 `PACKAGE_PIN` 和 `IOSTANDARD`。

## 3. Build：RTL → bitstream

从仓库根目录运行：

```bash
vivado -mode batch -nojournal \
  -log lab-hw-03-build.log \
  -source boards/kv260/scripts/build_lab03_marker.tcl
```

脚本目标 device 为 `xck26-sfvc784-2LV-c`，依次执行 synthesis、optimization、placement、routing、DRC report、timing report、bitstream generation。

成功时应生成：

- `build/kv260/lab-hw-03/kv260_marker_top.bit`
- `build/kv260/lab-hw-03/timing_summary.rpt`
- `build/kv260/lab-hw-03/utilization.rpt`
- `build/kv260/lab-hw-03/drc.rpt`

因为这个 design 故意没有 clock，正确 build 还必须打印 `TIMING_CHECK=NOT_APPLICABLE_CLOCKLESS`。这表示 timing check 明确“不适用”，不是宣称 timing closure。`STATUS=PASS` 仍然只说明 build helper 跑完；它还不是实体板 PASS。

## 4. 给 bitstream 一个不可含糊的身份

Linux/macOS shell：

```bash
sha256sum build/kv260/lab-hw-03/kv260_marker_top.bit
```

Windows PowerShell：

```powershell
Get-FileHash build/kv260/lab-hw-03/kv260_marker_top.bit -Algorithm SHA256
```

把 SHA-256 写进 evidence manifest。以后“这个 bitstream”指的是这个 hash 对应的 artifact，而不是某个同名文件。

## 5. Program：只把已经生成的 .bit 写进真实 XCK26

保持 LAB-HW-02 的供电与 J4 JTAG connection。

运行：

```bash
vivado -mode batch -nojournal \
  -log lab-hw-03-program.log \
  -source boards/kv260/scripts/program_bitstream.tcl \
  -tclargs build/kv260/lab-hw-03/kv260_marker_top.bit
```

program helper 会拒绝：

- bitstream 不存在；
- 参数不是 `.bit`；
- JTAG chain 中没有 `xck26*` device；
- `program_hw_devices` 报错。

成功 log 中应出现 `KV260_FPGA_DEVICE=xck26...` 和 `STATUS=PASS`.

## 6. Expected Evidence

program 成功后，观察 Bank 45 LED-class outputs 的稳定状态。

这里有一个刻意保留的事实边界：**在课程完成真实 KV260 dry run 之前，不把某个具体丝印 LED 的亮/灭极性写成已经验证的事实。** Board Store 支持的是 interface 与 pin mapping；真正的物理可见 designator/polarity 由本次上板记录。

通过 T-HW-003 至少需要：

- `lab-hw-03-build.log`；
- `timing_summary.rpt`、`utilization.rpt` 与 `drc.rpt`；
- build log 中的 `TIMING_CHECK=NOT_APPLICABLE_CLOCKLESS`；
- bitstream SHA-256；
- `lab-hw-03-program.log`；
- `xck26*` target identification；
- 真实板上的 marker observation / photo；
- carrier revision + Git commit。

### Save Evidence

复制 `boards/kv260/evidence/manifest.example.json`，建立本地 LAB-HW-03 evidence 文件并填写上述内容。

## 7. 为什么今天不能拿 DS34 当主要答案

UG1089 对 DS34 的定义是：**PS successfully loaded a PL design** 时点亮。

今天使用的是 development host → JTAG → direct PL programming。

所以今天的主 oracle 是：

**Vivado program result + XCK26 identity + 我们设计自己的 Bank 45 output。**

即使 DS34 有变化，也不要单独用它宣布 T-HW-003 PASS。

## 8. If it does not work

按层级排查：

1. **build 根本失败** → 先看 build log 的 synthesis / implementation error；
2. **bitstream 没生成** → 不要进入 programming；
3. **找不到 XCK26** → 回到 LAB-HW-02，检查 J12/J4/driver/JTAG；
4. **program 失败** → 保存 program log，不要改 neuron RTL；
5. **program 成功但看不到预期的 board-visible change** → 检查 carrier revision、XDC 与实际 board output；这是 physical mapping 问题，不是“再点一次 Program Device”就算完成。

## 9. Human Check

进入 LAB-HW-04 前，能解释：

1. synthesis、implementation、bitstream generation、programming 是哪四个不同阶段？
2. 为什么一个 `.bit` 文件需要 SHA-256？
3. 为什么 `bank45_gpio` 这个 RTL port 名本身不知道 J11 是什么？
4. 为什么 DS34 不能单独证明今天的 JTAG direct program？
5. 今天为什么故意不使用 clock、reset、AXI 或 neuron RTL？

## 10. 官方依据

- AMD UG1089 — Interfaces / J4 / reset / DS34  
  https://docs.amd.com/r/en-US/ug1089-kv260-starter-kit/Interfaces
- AMD/Xilinx Board Store — KV260 carrier `bank45_gpio`  
  https://github.com/Xilinx/XilinxBoardStore/blob/master/boards/Xilinx/kv260_carrier/1.3/board.xml
- AMD/Xilinx Board Store — K26 SOM package pin map  
  https://github.com/Xilinx/XilinxBoardStore/blob/master/boards/Xilinx/kv260_som/1.4/part0_pins.xml
- AMD DS987 — K26 SOM signal descriptions  
  https://docs.amd.com/r/en-US/ds987-k26-som/SOM240_1-Signal-Names-and-Descriptions